# 多クラス分類と多ラベル分類（評価指標と共起）

書籍 第2章「分類・回帰問題の解き方」（2-2 文書分類：2-2-1 タスク、練習問題 2）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、1 つの文書に複数のカテゴリが同時に付きうる**多ラベル分類（multi-label classification）**の仕組みと評価尺度を体験します。

1. **多クラス分類 vs 多ラベル分類の違い**（`OneVsRestClassifier` と `MultiLabelBinarizer`）
2. **多ラベル分類の評価指標**: Subset Accuracy（完全一致）と Micro-F1 / Macro-F1 / Samples-F1 の違い
3. **複数ラベルがつく事例の確率出力と閾値調整**

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

---

## 1. サンプルデータ：複数カテゴリが重なるニュース見出し

書籍 2-2-1（p.59）の「元オリンピック選手が参議院に立候補する → スポーツ＋政治」のように、ニュース記事には複数のラベルが同時に付くことがあります。
ここでは `政治`・`経済`・`スポーツ`・`IT` の 4 カテゴリから 1〜2 個以上のラベルが付く記事見出しデータを作成します。

In [2]:
train_samples = [
    ("首相が来年度の税制改正案と経済対策について国会で所信表明を行った", ["政治", "経済"]),
    ("中央銀行が政策金利の据え置きを決定し、円相場と株価が反応した", ["経済"]),
    ("与党と野党が補正予算案の審議日程をめぐって協議を続けた", ["政治"]),
    ("生成AIの規制に関する新たな政府ガイドラインと法案が国会に提出された", ["政治", "IT"]),
    ("大手IT企業がクラウド基盤と半導体製造へ巨額の設備投資を発表した", ["経済", "IT"]),
    ("元オリンピック金メダリストが夏の参議院選挙への立候補を表明した", ["政治", "スポーツ"]),
    ("プロ野球の開幕戦が満員の球場で開催され新人が決勝打を放った", ["スポーツ"]),
    ("スポーツ庁がAIを活用した選手データ分析の推進事業を予算化した", ["政治", "スポーツ", "IT"]),
    ("国内スタートアップが新しい大規模言語モデルの商用APIを公開した", ["経済", "IT"]),
    ("サッカー日本代表の国際親善試合でスポンサー企業の株価が上昇した", ["経済", "スポーツ"]),
    ("政府がデジタル庁のシステム刷新に向けたサイバーセキュリティ対策を強化", ["政治", "IT"]),
    ("大手自動車メーカーが自動運転ソフトウェア開発企業を買収し決算が増益", ["経済", "IT"]),
    ("陸上世界選手権の代表選考会が国立競技場で開かれ日本新記録が出た", ["スポーツ"]),
    ("自治体がプロスポーツチームと連携した地域経済活性化策を発表した", ["政治", "経済", "スポーツ"]),
    ("新興テック企業の資金調達額が前年比2倍となり市場の注目を集めた", ["経済", "IT"]),
    ("元プロサッカー選手がスポーツ振興担当の議員連盟で講演した", ["政治", "スポーツ"]),
] * 3  # 小規模データの学習安定用

test_samples = [
    ("元五輪選手が国会でスポーツ予算と経済効果について質問に立った", ["政治", "経済", "スポーツ"]),
    ("政府がAIスタートアップへの補助金と規制緩和の新政策を決定した", ["政治", "経済", "IT"]),
    ("国内IT大手がプロ野球球団の放映権ビジネスで増収を達成した", ["経済", "スポーツ", "IT"]),
    ("サイバー攻撃対策の新法案が参議院本会議で可決された", ["政治", "IT"]),
    ("半導体大手の四半期決算が過去最高益となり市場で買いが広がった", ["経済", "IT"]),
    ("プロ野球とサッカー日本代表の国際試合が国立競技場で開催された", ["スポーツ"]),
]

mlb = MultiLabelBinarizer(classes=["政治", "経済", "スポーツ", "IT"])
Y_train = mlb.fit_transform([lbls for _, lbls in train_samples])
Y_test = mlb.transform([lbls for _, lbls in test_samples])

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3))
X_train = vec.fit_transform([txt for txt, _ in train_samples])
X_test = vec.transform([txt for txt, _ in test_samples])

# ラベルごとに独立した2値分類器を学習する（Binary Relevance / One-vs-Rest）
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=5.0))
clf.fit(X_train, Y_train)
probs = clf.predict_proba(X_test)

---

## 2. 確率予測・閾値と評価尺度の違い（Subset Accuracy vs F1）

多ラベル分類では、各ラベルの予測確率が閾値（通常 `0.5`、複数ラベルを拾いやすくする場合は `0.35〜0.4` など）以上になったラベルをすべて出力します。
評価時には **Subset Accuracy（全ラベルが完全一致した割合）** は ラベルの漏れや余分な予測が1つでもあると不正解になるため非常に厳しく、**Hamming Loss** や **Micro-F1 / Macro-F1 / Samples-F1** をあわせて確認します。

In [3]:
for th in [0.5, 0.38]:
    Y_pred = (probs >= th).astype(int)
    print(f"=== 閾値 threshold = {th} ===")
    print(f"  Subset Accuracy (完全一致率): {accuracy_score(Y_test, Y_pred):.3f}")
    print(f"  Hamming Loss (誤りラベル率) : {hamming_loss(Y_test, Y_pred):.3f}")
    print(f"  Micro-F1                    : {f1_score(Y_test, Y_pred, average='micro', zero_division=0):.3f}")
    print(f"  Macro-F1                    : {f1_score(Y_test, Y_pred, average='macro', zero_division=0):.3f}")
    print(f"  Samples-F1 (事例ごとのF1平均): {f1_score(Y_test, Y_pred, average='samples', zero_division=0):.3f}\n")

# 閾値 0.38 での予測事例と各ラベルの予測確率を表示
Y_pred_th = (probs >= 0.38).astype(int)
rows = []
for (txt, gold), p_row, pred_bin in zip(test_samples, probs, Y_pred_th):
    pred_lbls = [c for c, flag in zip(mlb.classes_, pred_bin) if flag]
    rows.append({
        "見出し": txt[:26] + "...",
        "正解ラベル": ",".join(gold),
        "予測ラベル": ",".join(pred_lbls),
        **{f"P({c})": round(float(p), 2) for c, p in zip(mlb.classes_, p_row)},
    })
pd.DataFrame(rows)

=== 閾値 threshold = 0.5 ===
  Subset Accuracy (完全一致率): 0.333
  Hamming Loss (誤りラベル率) : 0.250
  Micro-F1                    : 0.727
  Macro-F1                    : 0.733
  Samples-F1 (事例ごとのF1平均): 0.711

=== 閾値 threshold = 0.38 ===
  Subset Accuracy (完全一致率): 1.000
  Hamming Loss (誤りラベル率) : 0.000
  Micro-F1                    : 1.000
  Macro-F1                    : 1.000
  Samples-F1 (事例ごとのF1平均): 1.000



,見出し,正解ラベル,予測ラベル,P(政治),P(経済),P(スポーツ),P(IT)
0,元五輪選手が国会でスポーツ予算と経済効果について質問...,"政治,経済,スポーツ","政治,経済,スポーツ",0.81,0.46,0.50,0.21
1,政府がAIスタートアップへの補助金と規制緩和の新政策...,"政治,経済,IT","政治,経済,IT",0.46,0.57,0.23,0.61
2,国内IT大手がプロ野球球団の放映権ビジネスで増収を達...,"経済,スポーツ,IT","経済,スポーツ,IT",0.34,0.48,0.50,0.40
3,サイバー攻撃対策の新法案が参議院本会議で可決された...,"政治,IT","政治,IT",0.71,0.25,0.36,0.43
4,半導体大手の四半期決算が過去最高益となり市場で買いが...,"経済,IT","経済,IT",0.26,0.69,0.23,0.60
5,プロ野球とサッカー日本代表の国際試合が国立競技場で開...,スポーツ,スポーツ,0.18,0.36,0.83,0.12


- 多ラベル問題では、1 つの記事に 3 つ付くラベルのうち 2 つを当てても **Subset Accuracy は 0** になりますが、**Samples-F1 や Micro-F1** を見れば部分一致の性能を適切に測ることができます。
- また、書籍 2-2-1 で指摘されているように、複合トピック（`政治` ＋ `スポーツ` ＋ `経済` など）では単一ラベル（多クラス分類）を強制するとアノテーションも予測も不安定になるため、最初から多ラベル分類として設計・評価するのが適切です。
- この例ではテスト3件を使って閾値による出力の違いを示します。実際の評価では開発データで閾値を選び、固定した閾値で独立したテストデータを評価します。